# Entrega 1 - Álgebra Linear

## Agentes inteligentes para o sucesso do estudante

O projeto reúne três funções em uma única solução:

- monitorar estudantes que podem precisar de acompanhamento;
- identificar pendências acadêmicas ou financeiras;
- orientar o estudante sobre a próxima ação.

Nesta entrega, os dados são transformados em vetores e matrizes. Em seguida, usamos **norma euclidiana** e **distância euclidiana**, duas operações de Álgebra Linear, para interpretar os perfis dos estudantes.

Os resultados servem somente como apoio. Eles não devem produzir decisões automáticas ou punitivas.

## Como executar este notebook

1. Instale o Python 3 e o Jupyter Notebook ou JupyterLab.
2. Mantenha este arquivo ao lado de uma pasta chamada `Base`.
3. Coloque os cinco arquivos Excel dentro da pasta `Base`, preservando seus nomes.
4. No terminal, dentro da pasta da entrega, instale as dependências com:

   `pip install pandas numpy openpyxl jupyter`

5. Inicie o Jupyter com `jupyter notebook` ou `jupyter lab`.
6. Abra este arquivo e escolha um kernel Python 3.
7. Use **Run All Cells** para executar todas as etapas.

Estrutura esperada:

```text
Entrega_Algebra_Linear/
├── Entrega_1_Algebra_Linear.ipynb
└── Base/
    ├── Contato.xlsx
    ├── Financeiro.xlsx
    ├── Historico.xlsx
    ├── Matriculas.xlsx
    └── Relacionamentos.xlsx
```

As saídas desta versão já estão salvas no notebook. Para reproduzir os resultados, é necessário ter acesso autorizado às cinco bases.

## 1. Preparação

O notebook procura a pasta `Base` ao lado do arquivo. Como alternativa, também verifica o diretório anterior e a pasta `Downloads/Base` do usuário. Os arquivos originais são apenas lidos e não são alterados.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.3f}")

nomes_arquivos = [
    "Contato.xlsx",
    "Matriculas.xlsx",
    "Financeiro.xlsx",
    "Historico.xlsx",
    "Relacionamentos.xlsx",
]

def possui_todas_as_bases(pasta):
    return pasta.exists() and all((pasta / nome).exists() for nome in nomes_arquivos)

candidatos = [
    Path("Base"),
    Path("."),
    Path("../Base"),
    Path(".."),
    Path.home() / "Downloads" / "Base",
]
downloads = Path.home() / "Downloads"
if downloads.exists():
    candidatos.extend(downloads.glob("*/Base"))
BASE_DIR = next((p.resolve() for p in candidatos if possui_todas_as_bases(p)), None)

if BASE_DIR is None:
    raise FileNotFoundError(
        "As cinco bases não foram encontradas. Crie uma pasta chamada Base ao lado do "
        "notebook e coloque nela: " + ", ".join(nomes_arquivos)
    )

arquivos = {
    "contato": BASE_DIR / "Contato.xlsx",
    "matriculas": BASE_DIR / "Matriculas.xlsx",
    "financeiro": BASE_DIR / "Financeiro.xlsx",
    "historico": BASE_DIR / "Historico.xlsx",
    "relacionamentos": BASE_DIR / "Relacionamentos.xlsx",
}

print(f"Bases localizadas em: {BASE_DIR}")

Bases localizadas em: C:\Users\sribe\Downloads\Entrega_Algebra_Linear\Base


## 2. Leitura e tratamento dos dados

Foram aplicados apenas tratamentos necessários para os cálculos:

- remoção de registros exatamente duplicados;
- exclusão de linhas financeiras sem informação de lançamento;
- conversão de datas e valores numéricos;
- seleção dos estudantes ativos em 2026-2;
- uso do histórico acadêmico até 2026-1, pois 2026-2 ainda está incompleto;
- agrupamento das informações para obter uma linha por estudante.

Dados demográficos, como sexo, idade, estado civil e localização, não entram nos cálculos.

In [2]:
# Matrículas: definição do grupo de estudantes ativos.
colunas_matriculas = [
    "ID_ALUNO",
    "Período Letivo Atual",
    "Status Acadêmico (Período Letivo Atual) (Período Letivo - Guideme)",
]
matriculas = pd.read_excel(arquivos["matriculas"], usecols=colunas_matriculas)
matriculas = matriculas.drop_duplicates()

status_ativos = {"Matriculado", "Pré-Matriculado", "Aguardando Rematrícula"}
col_status = "Status Acadêmico (Período Letivo Atual) (Período Letivo - Guideme)"

coorte = (
    matriculas.loc[
        (matriculas["Período Letivo Atual"].astype(str) == "2026-2")
        & (matriculas[col_status].isin(status_ativos)),
        ["ID_ALUNO"],
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Contato: usado somente para conferir a cobertura dos IDs.
contato = pd.read_excel(arquivos["contato"], usecols=["ID_ALUNO"])
contato = contato.drop_duplicates()

print(f"Estudantes ativos selecionados: {len(coorte):,}")
print(f"Estudantes com registro em Contato: {coorte['ID_ALUNO'].isin(contato['ID_ALUNO']).sum():,}")

Estudantes ativos selecionados: 4,000
Estudantes com registro em Contato: 4,000


In [3]:
# Financeiro: pendências vencidas e atraso médio em pagamentos.
colunas_financeiro = [
    "ID_ALUNO",
    "PERIODOLETIVO",
    "SERVICOPARCELA",
    "LANCAMENTOVALORORIGINAL",
    "STATUSLANCAMENTO",
    "LANCAMENTODATAVENCIMENTO",
    "LANCAMENTODATABAIXA",
]
financeiro = pd.read_excel(arquivos["financeiro"], usecols=colunas_financeiro)
linhas_financeiras_originais = len(financeiro)

campos_lancamento = [
    "SERVICOPARCELA",
    "LANCAMENTOVALORORIGINAL",
    "STATUSLANCAMENTO",
    "LANCAMENTODATAVENCIMENTO",
    "LANCAMENTODATABAIXA",
]
financeiro = financeiro.dropna(subset=campos_lancamento, how="all").drop_duplicates()
financeiro = financeiro[financeiro["ID_ALUNO"].isin(coorte["ID_ALUNO"])].copy()

financeiro["LANCAMENTODATAVENCIMENTO"] = pd.to_datetime(
    financeiro["LANCAMENTODATAVENCIMENTO"], errors="coerce"
)
financeiro["LANCAMENTODATABAIXA"] = pd.to_datetime(
    financeiro["LANCAMENTODATABAIXA"], errors="coerce"
)

data_referencia = pd.Timestamp("2026-08-31")
data_inicial_valida = pd.Timestamp("2020-01-01")

vencimento_valido = financeiro["LANCAMENTODATAVENCIMENTO"].between(
    data_inicial_valida, data_referencia
)
financeiro["PENDENCIA_VENCIDA"] = (
    financeiro["STATUSLANCAMENTO"].eq("Em Aberto")
    & vencimento_valido
    & (financeiro["LANCAMENTODATAVENCIMENTO"] <= data_referencia)
).astype(int)

status_pago = financeiro["STATUSLANCAMENTO"].isin(
    ["Baixado", "Baixado por Acordo", "Baixado parcialmente"]
)
data_baixa_valida = financeiro["LANCAMENTODATABAIXA"].between(
    data_inicial_valida, data_referencia
)
financeiro["DIAS_ATRASO"] = (
    financeiro["LANCAMENTODATABAIXA"] - financeiro["LANCAMENTODATAVENCIMENTO"]
).dt.days
financeiro.loc[~(status_pago & vencimento_valido & data_baixa_valida), "DIAS_ATRASO"] = np.nan
financeiro["DIAS_ATRASO"] = financeiro["DIAS_ATRASO"].clip(lower=0, upper=365)

fin_por_aluno = (
    financeiro.groupby("ID_ALUNO", as_index=False)
    .agg(
        Pendencias_Financeiras=("PENDENCIA_VENCIDA", "sum"),
        Dias_Atraso_Medio=("DIAS_ATRASO", "mean"),
    )
)

print(f"Linhas financeiras originais: {linhas_financeiras_originais:,}")
print(f"Linhas financeiras usadas na coorte: {len(financeiro):,}")

Linhas financeiras originais: 538,044
Linhas financeiras usadas na coorte: 123,695


In [4]:
# Histórico: média de notas, faltas e número de reprovações.
colunas_historico = [
    "ID_ALUNO",
    "Ano/Semestre Letivo",
    "Situação",
    "Nota Obtida",
    "Nota Distribuída",
    "Faltas Lançadas",
    "Aulas Ministradas",
]
historico = pd.read_excel(arquivos["historico"], usecols=colunas_historico)
linhas_historico_originais = len(historico)
historico = historico.drop_duplicates()
historico = historico[
    historico["ID_ALUNO"].isin(coorte["ID_ALUNO"])
    & (historico["Ano/Semestre Letivo"].astype(str) <= "2026-1")
].copy()

colunas_numericas = ["Nota Obtida", "Nota Distribuída", "Faltas Lançadas", "Aulas Ministradas"]
for coluna in colunas_numericas:
    historico[coluna] = pd.to_numeric(historico[coluna], errors="coerce")

situacoes_concluidas = {"Aprovado", "Reprovado", "Não Concluído"}
registro_concluido = historico["Situação"].isin(situacoes_concluidas)

nota_valida = registro_concluido & (historico["Nota Distribuída"] > 0)
historico["NOTA_PADRONIZADA"] = np.nan
historico.loc[nota_valida, "NOTA_PADRONIZADA"] = (
    10 * historico.loc[nota_valida, "Nota Obtida"]
    / historico.loc[nota_valida, "Nota Distribuída"]
).clip(0, 10)

faltas_validas = registro_concluido & (historico["Aulas Ministradas"] > 0)
historico["PERCENTUAL_FALTAS_CALCULADO"] = np.nan
historico.loc[faltas_validas, "PERCENTUAL_FALTAS_CALCULADO"] = (
    100 * historico.loc[faltas_validas, "Faltas Lançadas"]
    / historico.loc[faltas_validas, "Aulas Ministradas"]
).clip(0, 100)

historico["REPROVADO"] = historico["Situação"].eq("Reprovado").astype(int)

hist_por_aluno = (
    historico.groupby("ID_ALUNO", as_index=False)
    .agg(
        Media_Notas=("NOTA_PADRONIZADA", "mean"),
        Percentual_Faltas=("PERCENTUAL_FALTAS_CALCULADO", "mean"),
        Reprovacoes=("REPROVADO", "sum"),
    )
)

print(f"Linhas históricas originais: {linhas_historico_originais:,}")
print(f"Linhas históricas após retirar duplicações e selecionar a coorte: {len(historico):,}")

Linhas históricas originais: 274,901
Linhas históricas após retirar duplicações e selecionar a coorte: 86,661


In [5]:
# Relacionamentos: quantidade de registros explicitamente classificados como risco de evasão.
colunas_relacionamento = ["ID_ALUNO", "Código", "Tema Relacionamento", "Data de Criação"]
relacionamentos = pd.read_excel(arquivos["relacionamentos"], usecols=colunas_relacionamento)
relacionamentos = relacionamentos.drop_duplicates(subset=["Código"])
relacionamentos["Data de Criação"] = pd.to_datetime(
    relacionamentos["Data de Criação"], errors="coerce"
)
relacionamentos = relacionamentos[
    relacionamentos["ID_ALUNO"].isin(coorte["ID_ALUNO"])
    & (relacionamentos["Data de Criação"] <= data_referencia)
].copy()
relacionamentos["REGISTRO_RISCO"] = relacionamentos["Tema Relacionamento"].eq(
    "Risco de Evasão"
).astype(int)

rel_por_aluno = (
    relacionamentos.groupby("ID_ALUNO", as_index=False)
    .agg(Registros_Risco=("REGISTRO_RISCO", "sum"))
)

print(f"Registros de relacionamento usados: {len(relacionamentos):,}")

Registros de relacionamento usados: 42,314


## 3. Base final por estudante

As bases são reunidas pelo `ID_ALUNO`. Ausência de pendência ou registro de risco é representada por zero. Ausência de histórico acadêmico permanece como dado ausente, pois não significa nota ou frequência zero.

In [6]:
base_estudantes = (
    coorte.merge(hist_por_aluno, on="ID_ALUNO", how="left")
    .merge(fin_por_aluno, on="ID_ALUNO", how="left")
    .merge(rel_por_aluno, on="ID_ALUNO", how="left")
)

colunas_zero = [
    "Reprovacoes",
    "Pendencias_Financeiras",
    "Dias_Atraso_Medio",
    "Registros_Risco",
]
base_estudantes[colunas_zero] = base_estudantes[colunas_zero].fillna(0)
base_estudantes["Historico_Disponivel"] = base_estudantes[
    ["Media_Notas", "Percentual_Faltas"]
].notna().all(axis=1)

resumo_base = pd.DataFrame(
    {
        "Indicador": [
            "Estudantes ativos",
            "Com histórico suficiente",
            "Sem histórico suficiente",
        ],
        "Quantidade": [
            len(base_estudantes),
            int(base_estudantes["Historico_Disponivel"].sum()),
            int((~base_estudantes["Historico_Disponivel"]).sum()),
        ],
    }
)
resumo_base

,Indicador,Quantidade
0,Estudantes ativos,4000
1,Com histórico suficiente,3383
2,Sem histórico suficiente,617


## 4. Vetores e matriz

Cada estudante com histórico suficiente será representado pelo vetor:

\[
v_i = [r_{nota}, r_{faltas}, r_{reprovações}, r_{pendências}, r_{atraso}, r_{relacionamento}]
\]

Todos os valores ficam entre 0 e 1 e apontam na mesma direção: valores maiores indicam sinais mais fortes de necessidade de acompanhamento.

Para evitar que poucos valores extremos dominem o cálculo, contagens e dias de atraso são limitados pelo percentil 95 antes da normalização.

In [7]:
def normalizar_positivo(serie):
    serie = pd.to_numeric(serie, errors="coerce").fillna(0).clip(lower=0)
    limite = serie.quantile(0.95)
    if limite <= 0:
        return pd.Series(0.0, index=serie.index)
    return serie.clip(upper=limite) / limite


analise = base_estudantes[base_estudantes["Historico_Disponivel"]].copy()

analise["Risco_Nota"] = (1 - analise["Media_Notas"] / 10).clip(0, 1)
analise["Risco_Faltas"] = (analise["Percentual_Faltas"] / 100).clip(0, 1)
analise["Risco_Reprovacoes"] = normalizar_positivo(analise["Reprovacoes"])
analise["Risco_Pendencias"] = normalizar_positivo(analise["Pendencias_Financeiras"])
analise["Risco_Atraso"] = normalizar_positivo(analise["Dias_Atraso_Medio"])
analise["Risco_Relacionamento"] = normalizar_positivo(analise["Registros_Risco"])

colunas_vetor = [
    "Risco_Nota",
    "Risco_Faltas",
    "Risco_Reprovacoes",
    "Risco_Pendencias",
    "Risco_Atraso",
    "Risco_Relacionamento",
]

X = analise[colunas_vetor].to_numpy(dtype=float)

print(f"Dimensão da matriz X: {X.shape[0]} estudantes x {X.shape[1]} características")
print("Exemplo de vetor:")
print(np.round(X[0], 3))

Dimensão da matriz X: 3383 estudantes x 6 características
Exemplo de vetor:
[0.366 0.    0.091 0.    1.    0.   ]


## 5. Operação 1: norma euclidiana

A norma mede o tamanho do vetor de cada estudante:

\[
\|v_i\|_2 = \sqrt{x_1^2 + x_2^2 + \cdots + x_6^2}
\]

Neste contexto, uma norma maior significa que o estudante apresenta uma combinação mais intensa de sinais. A norma não explica sozinha a situação; por isso, também identificamos qual dimensão possui o maior valor.

In [8]:
analise["Norma"] = np.linalg.norm(X, axis=1)

nomes_dimensoes = {
    "Risco_Nota": "nota",
    "Risco_Faltas": "faltas",
    "Risco_Reprovacoes": "reprovações",
    "Risco_Pendencias": "pendências financeiras",
    "Risco_Atraso": "atraso financeiro",
    "Risco_Relacionamento": "relacionamento",
}
analise["Dimensao_Principal"] = (
    analise[colunas_vetor].idxmax(axis=1).map(nomes_dimensoes)
)

maiores_normas = analise.nlargest(10, "Norma")[
    ["ID_ALUNO", "Norma", "Dimensao_Principal"]
].reset_index(drop=True)
maiores_normas

,ID_ALUNO,Norma,Dimensao_Principal
0,11314,2.090,reprovações
1,89,1.948,pendências financeiras
2,17773,1.940,reprovações
3,3357,1.937,reprovações
4,2377,1.911,reprovações
5,9527,1.888,pendências financeiras
6,15243,1.871,pendências financeiras
7,3480,1.869,reprovações
8,9984,1.863,reprovações
9,12720,1.853,reprovações


### Interpretação da norma

- O agente de monitoramento pode usar a norma para organizar uma fila inicial de análise.
- O agente de pendências consulta a dimensão principal para entender o motivo da prioridade.
- O agente para o estudante transforma a dimensão identificada em uma orientação clara.

A análise humana continua necessária antes de qualquer contato ou decisão.

## 6. Operação 2: distância euclidiana

A distância entre dois estudantes é calculada por:

\[
d(v_i,v_j) = \sqrt{\sum_{k=1}^{6}(v_{ik}-v_{jk})^2}
\]

Quanto menor a distância, mais parecidos são os perfis nas seis características analisadas.

In [9]:
# Exemplo: usamos como referência o estudante com maior norma.
indice_referencia = analise["Norma"].idxmax()
vetor_referencia = analise.loc[indice_referencia, colunas_vetor].to_numpy(dtype=float)

analise["Distancia_Referencia"] = np.linalg.norm(X - vetor_referencia, axis=1)

similares = (
    analise.loc[analise.index != indice_referencia]
    .nsmallest(5, "Distancia_Referencia")
    [["ID_ALUNO", "Distancia_Referencia", "Norma", "Dimensao_Principal"]]
    .reset_index(drop=True)
)

print(f"Estudante de referência: {int(analise.loc[indice_referencia, 'ID_ALUNO'])}")
similares

Estudante de referência: 11314


,ID_ALUNO,Distancia_Referencia,Norma,Dimensao_Principal
0,89,0.320,1.948,pendências financeiras
1,17773,0.336,1.940,reprovações
2,3357,0.343,1.937,reprovações
3,2377,0.368,1.911,reprovações
4,17666,0.392,1.843,reprovações


### Interpretação da distância

Perfis próximos podem receber checklists e orientações iniciais semelhantes. Por exemplo, estudantes com faltas elevadas e várias reprovações podem ser encaminhados para apoio acadêmico. Já estudantes com pendências e atrasos podem receber orientação sobre o setor financeiro.

A semelhança não significa que os casos são idênticos. O histórico e o contexto individual ainda precisam ser verificados.

## 7. Conclusão

Os dados acadêmicos, financeiros e de relacionamento foram representados como vetores. A reunião desses vetores produziu uma matriz de estudantes por características.

A norma permitiu medir a intensidade conjunta dos sinais de acompanhamento. A distância permitiu encontrar estudantes com perfis semelhantes. Assim, a Álgebra Linear ajuda os três agentes a organizar casos, identificar a principal dimensão de atenção e preparar uma orientação inicial.

As medidas não devem ser usadas como decisão final. Dados ausentes, regras institucionais e avaliação humana precisam ser considerados antes de qualquer ação.